# Combine all of the ICD10 codes with the NDD cases and controls

- Generated from the Pull and Prep scripts previously.

In [1]:
import pandas as pd
import numpy as np

In [2]:
# Select your NDD and the date
ndd = 'VAS'
date = 'JULY_23_2026'

In [3]:
#! pwd

In [ ]:
# Select the NDD case files created in step 01
# cases = pd.read_csv(f'/data/DEM_with_tenure_n987_JULY_23_2026.csv')
# cases = pd.read_csv(f'/data/PD_with_tenure_n442_JULY_23_2026.csv')
cases = pd.read_csv(f'/data/VAS_with_tenure_n177_JULY_23_2026.csv')
# cases = pd.read_csv(f'/data/AD_with_tenure_n316_JULY_23_2026.csv')
cases

In [ ]:
#Load controls created in step 02
controls = pd.read_csv('/data/CONTROLS_with_tenure_n150638_JULY_23_2026.csv')
controls = controls.drop(columns = 'AD_DATE')
controls

In [ ]:
# Combine cases and controls
df = pd.concat([cases, controls])

#Check to make sure no duplicate IDs
# print(df.ID.value_counts())

df = df.sort_values(by = f'{ndd}_DATE')
df = df.drop_duplicates(subset = 'ID', keep = 'first')

#Check to make sure no duplicate IDs
# print(df.ID.value_counts())

df

In [ ]:
#Check number of cases and controls
df[f'{ndd}_DATE'].isna().value_counts()

# Add ICD10 Codes

In [ ]:
codes = pd.read_csv('../../data/labels.csv')
codes = codes[['FinnGen_Phenocode','ICD10_Codes','Cohort','Type','UKB_Description_For_Plots']]
codes = codes[codes['Cohort']=='UKB']
codes

In [ ]:
t = codes['FinnGen_Phenocode'].str.split(',').explode().str.strip().tolist()
t[:10]

In [ ]:
condition_list = t.copy()
condition = 'J10_INFLUPNEU'
test = pd.read_csv(f'/data/ICD10_Codes/Finngen_codes/{condition}.csv')
test = test[['ID',condition]]
test

In [ ]:
for condition in condition_list:
    try:
        test = pd.read_csv(f'/data/ICD10_Codes/Finngen_codes/{condition}.csv')
        # test = test.rename(columns = {'person_id':'ID', 'start_date': condition})
        test = test[['ID', condition]]
        df = df.merge(test, left_on = 'ID', right_on = 'ID', how = 'left')
    except Exception as e:
        print(f'CANNOT FIND INDIVIDUAL DATA WITH {condition} IN FILES')
        continue

In [ ]:
ls -l ./data/ICD10_Codes/Finngen_codes | wc -l 

In [ ]:
df[f'{ndd}_DATE'].isna().value_counts()

In [16]:
#Encode NDD to 1 or 0
df[ndd] = np.where(df[ndd + '_DATE'].isna(), 0, 1)

#GENETIC_SEX to 1 or 2
df.loc[df.sex_at_birth == 'Female', 'SEX'] = '0'
df.loc[df.sex_at_birth == 'Male', 'SEX'] = '1'

In [ ]:
df.columns

In [ ]:
df

In [ ]:
# Because we only have reliable virus data since 2015, the longer the study could be is 9 years
START_DATE = '2015-01-01'

for code in condition_list:
    print(code)
    try:
        df[f'QC0_{code}'] = np.where((pd.to_datetime(df[f'{code}']) < pd.to_datetime(df['tenure_date'])), 1, 0)
        df[f'{code}'] = np.where((pd.to_datetime(df[f'{code}']) < pd.to_datetime(df['tenure_date'])), 1, 0)

    except Exception as e:
        print(f'CANNOT FIND {code} INFO')

In [ ]:
df

In [ ]:
df.QC0_J10_INFLUPNEU.value_counts()

# Add genetic status

# add apoe

In [ ]:
apoe = pd.read_csv('/data/other/APOE_genotypes.csv')
#eliminate unknown samples
apoe = apoe[apoe['APOE_GENOTYPE'] != 'unknown']
apoe

In [ ]:
apoe.APOE_GENOTYPE.value_counts()

In [27]:
apoe["APOE"] = apoe["APOE_GENOTYPE"].map({
    "e3/e4": 1,
    "e4/e4": 2
}).fillna(0).astype(int)

In [ ]:
apoe = apoe[['IID', 'APOE']]
apoe = apoe.rename(columns = {'IID':'ID'})
apoe

In [ ]:
df = df.merge(apoe, left_on = 'ID', right_on = 'ID', how = 'left')
df

In [ ]:
df.APOE.value_counts(dropna=False)

In [ ]:
df = df[~df['APOE'].isna()]
df

In [32]:
date = 'JULY_23_2026'
df.to_csv(f'/data/{ndd}_{date}_ready_cox.csv', header=True, index=False)